# 02 · The one `einsum` behind every KAN edge function

Every edge in our KAN layers is evaluated by a **single line**:

```python
edge = torch.einsum("bik,oik->boi", psi, self.w_rbf)   # RBFEdgeFunctions.forward
```

`AddKANLayer` then just sums those edges over the inputs, and `LeanKANLayer`
multiplies some and sums the rest — so if you understand *this* contraction, you
understand the whole forward pass.

This notebook takes **3 real biodiesel trajectories**, walks the einsum index by
index, and reproduces it three independent ways (triple `for`-loop, broadcasting,
`matmul`) so nothing is a black box.

**Roadmap**
0. `einsum` in **generic** index notation (`i j k …`) with a tiny toy — no chemistry
1. **Map** those neutral letters onto our project's `batch / input / output / k`
2. Build the real inputs `psi` and `w_rbf` from data
3. Run the einsum, then re-derive it by hand and assert equality
4. See how `AddKAN` (sum) and `LeanKAN` (product+sum) sit on top of it


## Paper-first roadmap: the actual biodiesel ChemKAN

Before touching `einsum`, pin the calculation to the architecture in the ChemKAN paper.

For biodiesel there are six species and one temperature input,

$$
\mathbf{x}_0=\mathbf{u} = [\mathrm{TG},\mathrm{ROH},\mathrm{DG},\mathrm{MG},\mathrm{GL},\mathrm{R'CO_2R},T]^\top \in\mathbb{R}^{7}
$$

The kinetic core is the two-layer composition from ChemKAN Eq. 16,

$$
\mathrm{KAN}_{\mathrm{kin}}
=
\Psi^{\mathrm{lean}}_{1}\circ\Psi^{\mathrm{add}}_{0}.
$$

For the biodiesel architecture:

$$
\boxed{7 \xrightarrow{\;\Psi^{\mathrm{add}}_0\;} 4
       \xrightarrow{\;\Psi^{\mathrm{lean}}_1,\; n^{\mathrm{mu}}=2\;} 6}
$$

- $n_0=7$: the six species plus temperature.
- $n_1=4$: the four hidden nodes reported in the paper.
- $n_2=6$: the six species-rate outputs.
- Every activation uses a three-point RBF grid, so $N=3$.

The final six numbers are

$$
\dot{\tilde{\mathbf u}}
=
\left[
\frac{d\mathrm{TG}}{dt},
\frac{d\mathrm{ROH}}{dt},
\frac{d\mathrm{DG}}{dt},
\frac{d\mathrm{MG}}{dt},
\frac{d\mathrm{GL}}{dt},
\frac{d\mathrm{R'CO_2R}}{dt}
\right]^\top ,
$$

which is the biodiesel instance of ChemKAN Eq. 13.


## 0A · The paper's indices: $ (l,\alpha,\beta) \text{ and the matrix } (\Phi_l)$

The paper writes a KAN layer as a **matrix of univariate functions**,

$$
\Phi_l =
\begin{bmatrix}
\phi_{l,1,1}(\cdot) & \phi_{l,1,2}(\cdot) & \cdots & \phi_{l,1,n_l}(\cdot)\\
\phi_{l,2,1}(\cdot) & \phi_{l,2,2}(\cdot) & \cdots & \phi_{l,2,n_l}(\cdot)\\
\vdots & \vdots & \ddots & \vdots\\
\phi_{l,n_{l+1},1}(\cdot) & \phi_{l,n_{l+1},2}(\cdot) & \cdots &
\phi_{l,n_{l+1},n_l}(\cdot)
\end{bmatrix}.
$$

The meaning of the indices is:

| Paper symbol | Meaning | Code counterpart |
|---|---|---|
| $l$ | which KAN layer | implicit in the PyTorch module instance |
| $\alpha$ | **destination/output node** in layer $l+1$ | `o` |
| $\beta$ | **source/input node** in layer $l$ | `i` |
| $N$ | number of RBF basis functions on one edge | `K` / `num_basis` |
| paper's basis index in Eq. 11 | which RBF bump on that edge | `k` |
| batch index | not shown in the paper's single-sample equations | `b` |

So the paper's example “$\alpha=3,\beta=2$” means:

> the edge from **input node 2** of layer $l$ to **output node 3** of layer $l+1$.

### Rows and columns of $\Phi_l$

- A **row $\alpha$** contains every edge function that enters destination node $\alpha$.
- A **column $\beta$** contains every edge function leaving source node $\beta$.

This is the opposite of thinking of $\Phi_l$ as a table of ordinary scalar weights.  
Every entry is itself a learned function $\phi_{l,\alpha,\beta}(\cdot)$.


## 0B · First biodiesel layer: $(\Phi_0)$ is a $(4\times7)$ function matrix

For $l=0$,

$$
n_0=7,\qquad n_1=4.
$$

Therefore,

$$
\Phi_0 \in
\left(\text{univariate functions}\right)^{4\times7}.
$$

Writing the columns with their physical meanings,

$$
\Phi_0 =
\begin{bmatrix}
\phi_{0,1,\mathrm{TG}} & \phi_{0,1,\mathrm{ROH}} & \phi_{0,1,\mathrm{DG}} &
\phi_{0,1,\mathrm{MG}} & \phi_{0,1,\mathrm{GL}} & \phi_{0,1,\mathrm{R'CO_2R}} & \phi_{0,1,T}\\
\phi_{0,2,\mathrm{TG}} & \phi_{0,2,\mathrm{ROH}} & \phi_{0,2,\mathrm{DG}} &
\phi_{0,2,\mathrm{MG}} & \phi_{0,2,\mathrm{GL}} & \phi_{0,2,\mathrm{R'CO_2R}} & \phi_{0,2,T}\\
\phi_{0,3,\mathrm{TG}} & \phi_{0,3,\mathrm{ROH}} & \phi_{0,3,\mathrm{DG}} &
\phi_{0,3,\mathrm{MG}} & \phi_{0,3,\mathrm{GL}} & \phi_{0,3,\mathrm{R'CO_2R}} & \phi_{0,3,T}\\
\phi_{0,4,\mathrm{TG}} & \phi_{0,4,\mathrm{ROH}} & \phi_{0,4,\mathrm{DG}} &
\phi_{0,4,\mathrm{MG}} & \phi_{0,4,\mathrm{GL}} & \phi_{0,4,\mathrm{R'CO_2R}} & \phi_{0,4,T}
\end{bmatrix}.
$$

There are therefore

$$
4\times7=28
$$

distinct edge functions in the first layer.

For example,

$$
\phi_{0,3,2}(x_{0,2})
$$

is the value carried on the edge

$$
\boxed{\text{ROH input node }2 \longrightarrow \text{hidden node }3}.
$$

### What Eq. 11 does on one of those 28 edges

For any fixed $(l,\alpha,\beta)$,

$$
\phi_{l,\alpha,\beta}(x_{l,\beta})
=
\sum_{k=1}^{N}
w^\psi_{l,\alpha,\beta,k}
\psi_k(x_{l,\beta})
+
w^b_{l,\alpha,\beta}b(x_{l,\beta}).
$$

For biodiesel $N=3$. Ignoring the optional base path for the moment,

$$
\phi_{0,3,2}(x_{0,2})
=
w^\psi_{0,3,2,1}\psi_1(x_{0,2})
+
w^\psi_{0,3,2,2}\psi_2(x_{0,2})
+
w^\psi_{0,3,2,3}\psi_3(x_{0,2}).
$$

**This is where the basis index disappears.**  
The three basis contributions are summed to obtain **one scalar edge value**.

Important wording:

$$
\boxed{\text{sum over }k \Rightarrow \text{one edge value, not yet one node value}}
$$

The hidden-node value comes in the next step.


> **Important:** when the paper writes a compact expression such as $(\mathbf y=\Phi_l \mathbf{x})$,
> $(\Phi_l)$ is **not an ordinary numerical weight matrix**. Its entries are functions.
> Operationally, each entry first evaluates
> $(\phi_{l,\alpha,\beta}(x_{l,\beta}))$, and then the row is aggregated.
> For AddKAN that row aggregation is a sum over $(\beta)$.


## 0C · AddKAN: after summing the 3 basis functions, sum the 7 incoming edges

Define the evaluated edge value

$$
E^{(0)}_{\alpha,\beta}
=
\phi_{0,\alpha,\beta}(x_{0,\beta}).
$$

For hidden node $\alpha$, AddKAN sums the seven incoming edge values:

$$
x_{1,\alpha}
=
\sum_{\beta=1}^{7}
E^{(0)}_{\alpha,\beta}
=
\sum_{\beta=1}^{7}
\phi_{0,\alpha,\beta}(x_{0,\beta}).
$$

For hidden node 3 specifically,

$$
\begin{aligned}
x_{1,3}
=&\;
\phi_{0,3,1}(x_{0,1})
+\phi_{0,3,2}(x_{0,2})
+\cdots
+\phi_{0,3,7}(x_{0,7})\\
=&\;
\phi_{0,3,\mathrm{TG}}(\mathrm{TG})
+\phi_{0,3,\mathrm{ROH}}(\mathrm{ROH})
+\cdots
+\phi_{0,3,T}(T).
\end{aligned}
$$

So there are **two different sums**:

1. **inside each edge:** sum over the 3 basis functions $k$;
2. **inside AddKAN's node:** sum over the 7 source nodes $\beta$.

After doing this for all four rows of $\Phi_0$,

$$
\mathbf{x}_1=[x_{1,1},x_{1,2},x_{1,3},x_{1,4}]^\top\in\mathbb{R}^{4}.
$$


## 0D · Second biodiesel layer: $\Phi_1$ is $6\times4$, then LeanKAN uses $n^{\mathrm{mu}}=2$

Now the four hidden values are the next layer's inputs:

$$
\mathbf{x}_1\in\mathbb{R}^{4}.
$$

The second layer has six outputs, so

$$
\Phi_1 =
\begin{bmatrix}
\phi_{1,1,1} & \phi_{1,1,2} & \phi_{1,1,3} & \phi_{1,1,4}\\
\phi_{1,2,1} & \phi_{1,2,2} & \phi_{1,2,3} & \phi_{1,2,4}\\
\vdots & \vdots & \vdots & \vdots\\
\phi_{1,6,1} & \phi_{1,6,2} & \phi_{1,6,3} & \phi_{1,6,4}
\end{bmatrix},
$$

a $6\times4$ matrix of 24 learned edge functions.

Again, each one of those 24 edge functions is internally made from the three RBF basis functions.

But now the layer is **LeanKAN**, not AddKAN. With $n^{\mathrm{mu}}=2$, for every destination/output node $\alpha$,

$$
y^{\mathrm{mult}}_{1,\alpha}
=
\prod_{\beta=1}^{2}
\phi_{1,\alpha,\beta}(x_{1,\beta}),
$$

$$
y^{\mathrm{add}}_{1,\alpha}
=
\sum_{\beta=3}^{4}
\phi_{1,\alpha,\beta}(x_{1,\beta}),
$$

and

$$
x_{2,\alpha}
=
y^{\mathrm{mult}}_{1,\alpha}
+
y^{\mathrm{add}}_{1,\alpha}.
$$

Therefore, for output node $\alpha=1$,

$$
x_{2,1}
=
\phi_{1,1,1}(x_{1,1})\,
\phi_{1,1,2}(x_{1,2})
+
\phi_{1,1,3}(x_{1,3})
+
\phi_{1,1,4}(x_{1,4}).
$$

In the biodiesel ordering, $x_{2,1}$ is the first species rate, e.g. $d\mathrm{TG}/dt$.

The same structure is evaluated independently for all six rows, producing

$$
\mathbf{x}_2
=
\dot{\tilde{\mathbf u}}
\in\mathbb{R}^{6}.
$$


## 0E · Now the PyTorch `einsum` has an exact paper meaning

For a batch of states, the paper's single-sample edge equation becomes

$$
E^{(l)}_{b,\alpha,\beta}
=
\sum_{k=1}^{N}
\psi_{b,\beta,k}\,
w^\psi_{l,\alpha,\beta,k}.
$$

Our code writes the same thing as

```python
edge = torch.einsum("bik,oik->boi", psi, w_rbf)
```

with the mapping

$$
o\leftrightarrow\alpha,\qquad
i\leftrightarrow\beta,\qquad
k\leftrightarrow\text{paper's RBF/grid index}.
$$

For the first biodiesel layer:

- `psi.shape = (B, 7, 3)`
- `w_rbf.shape = (4, 7, 3)`
- `edge.shape = (B, 4, 7)`

and then AddKAN does

```python
hidden = edge.sum(dim=-1)
```

which is exactly

$$
x_{1,\alpha}=\sum_{\beta=1}^{7}E^{(0)}_{\alpha,\beta}.
$$

For the second layer:

- `psi.shape = (B, 4, 3)`
- `w_rbf.shape = (6, 4, 3)`
- `edge.shape = (B, 6, 4)`

and LeanKAN combines the four retained $\beta$-indexed edge values with the product/sum rule above.

### Why is the layer index $l$ missing from `w_rbf[o,i,k]`?

Because each PyTorch layer object owns its own `w_rbf` tensor.

Conceptually, the full symbol is

$$
w^\psi_{l,\alpha,\beta,k},
$$

but in code the layer object already tells us $l$.  
So inside one module it is enough to store

```python
w_rbf[o, i, k]
```

rather than a single global `w_rbf[l, o, i, k]`.


> **Notation warning from the papers:** the same letters are reused in different equations.
>
> - In the RBF expansion, the ChemKAN paper uses $i$ as the **basis/grid index**.
> - In the LeanKAN equations, $i$ is used as an **output-node index** and $j$ as the **input-node index**.
> - In our PyTorch code, `i` means **input node** and `k` means **basis function**.
>
> The notebook therefore uses `o`, `i`, `k` deliberately because it is less ambiguous than copying every paper letter literally.


## 0 · `einsum` in plain, generic index notation

Forget chemistry for a moment. `torch.einsum` is just **Einstein summation**: you label
the axes of each input tensor with letters, write an arrow, and list the letters you want
in the output. Two rules cover everything:

1. **A letter missing from the output is summed over** (contracted / "dummy" index).
2. **A letter kept in the output is a free axis** — and if it appears in *both* inputs it
   is aligned element-by-element (a *batched* axis), not contracted.

Some familiar operations in this one notation (using neutral letters `i j k`):

| operation | einsum | meaning |
|:--|:--|:--|
| matrix multiply | `"ij,jk->ik"` | $C_{ik}=\sum_j A_{ij}B_{jk}$ — `j` summed |
| dot product | `"i,i->"` | $\sum_i a_i b_i$ — everything summed |
| outer product | `"i,j->ij"` | $a_i b_j$ — nothing summed |
| transpose | `"ij->ji"` | just relabel axes |
| row sum | `"ij->i"` | $\sum_j A_{ij}$ |
| **batched** matmul | `"bij,bjk->bik"` | `b` kept & aligned, `j` summed |

The last row is the key pattern: `b` appears in both inputs **and** the output, so it is a
*batch* axis that lines up one-to-one; `j` appears in both inputs but **not** the output,
so it is *contracted*. Our KAN einsum is exactly this shape with one extra kept axis.


### A 2×2 toy — read the rule off the numbers


In [ ]:
import torch
torch.set_printoptions(precision=4, sci_mode=False)

A = torch.tensor([[1., 2.],
                  [3., 4.]])

print("Shape A: ", A.shape)

B = torch.tensor([[5., 6.],
                  [7., 8.]])

print("Shape B: ", B.shape)

# "ij,jk->ik": sum over j (shared, dropped); keep i and k
C = torch.einsum("ij,jk->ik", A, B)
print("matmul  ij,jk->ik :\n", C, "\n== A@B ?", torch.equal(C, A @ B))


# same data, but keep j too -> "ij,jk->ijk" contracts NOTHING, just broadcasts
full = torch.einsum("ij,jk->ijk", A, B)
print("\nkeep j  ij,jk->ijk shape:", tuple(full.shape),
      "  and sum over j gives back C ?",
      torch.equal(full.sum(dim=1), C))


Shape A:  torch.Size([2, 2])
Shape B:  torch.Size([2, 2])
matmul  ij,jk->ik :
 tensor([[19., 22.],
        [43., 50.]]) 
== A@B ? True

keep j  ij,jk->ijk shape: (2, 2, 2)   and sum over j gives back C ? True


In [37]:
result = torch.tensor(torch.zeros(2,2))
for i in range(A.shape[0]):
    for k in range(B.shape[1]):
        for j in range(B.shape[0]):
            print("==================")
            print(f" i:{i},\n k:{k}, \n j:{j}")
            print(f"(i,j) = {(i,j)}, and (j,k) = {(j,k)}")
            print("==================")
            print(f"A[{i},{j}] is", A[i,j].numpy())
            print(f"B[{j},{k}] is", B[j,k].numpy(), "\n")
            result[i, k] += A[i,j] * B[j,k]
        
        print("-----Sum-----")
        print(f"Result{[i,k]} = {result[i,k]}")
        print("-------------\n")

print(result)

 i:0,
 k:0, 
 j:0
(i,j) = (0, 0), and (j,k) = (0, 0)
A[0,0] is 1.0
B[0,0] is 5.0 

 i:0,
 k:0, 
 j:1
(i,j) = (0, 1), and (j,k) = (1, 0)
A[0,1] is 2.0
B[1,0] is 7.0 

-----Sum-----
Result[0, 0] = 19.0
-------------

 i:0,
 k:1, 
 j:0
(i,j) = (0, 0), and (j,k) = (0, 1)
A[0,0] is 1.0
B[0,1] is 6.0 

 i:0,
 k:1, 
 j:1
(i,j) = (0, 1), and (j,k) = (1, 1)
A[0,1] is 2.0
B[1,1] is 8.0 

-----Sum-----
Result[0, 1] = 22.0
-------------

 i:1,
 k:0, 
 j:0
(i,j) = (1, 0), and (j,k) = (0, 0)
A[1,0] is 3.0
B[0,0] is 5.0 

 i:1,
 k:0, 
 j:1
(i,j) = (1, 1), and (j,k) = (1, 0)
A[1,1] is 4.0
B[1,0] is 7.0 

-----Sum-----
Result[1, 0] = 43.0
-------------

 i:1,
 k:1, 
 j:0
(i,j) = (1, 0), and (j,k) = (0, 1)
A[1,0] is 3.0
B[0,1] is 6.0 

 i:1,
 k:1, 
 j:1
(i,j) = (1, 1), and (j,k) = (1, 1)
A[1,1] is 4.0
B[1,1] is 8.0 

-----Sum-----
Result[1, 1] = 50.0
-------------

tensor([[19., 22.],
        [43., 50.]])


/var/folders/b_/hzz32tks56n7j4ymt8wvxr940000gn/T/ipykernel_31741/4049814665.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  result = torch.tensor(torch.zeros(2,2))


That second example is the whole intuition: **keeping** a shared letter vs.
**dropping** it is the only difference between a broadcast and a contraction. `einsum`
just does the broadcast-multiply and then sums away exactly the letters you left out.


## 1 · Map the generic letters onto the project's notation

Now rename the neutral letters to what they mean in a KAN layer. Our edge einsum is
`"bik,oik->boi"` — structurally a batched matmul with an **extra kept axis** `i`:

| generic role | our letter | axis meaning | biodiesel size |
|:--|:--:|:--|:--:|
| batch (aligned, kept) | `b` | trajectory in the batch | 2 (we take 2) |
| **contracted** (shared, dropped) | `k` | Gaussian basis-bump index | 3 (`num_basis`) |
| kept, shared across both inputs | `i` | input feature (the $m{+}1=7$ vars $[Y,T]$) | 7 |
| kept, from weights only | `o` | output node (hidden unit) | 4 (`hidden_dim`) |

Read side by side with the toy:

- generic `"bij,bjk->bik"` contracts `j`; ours `"bik,oik->boi"` contracts **`k`**.
- generic keeps batch `b`; ours keeps batch `b` **and** input `i` (both are aligned,
  element-for-element, between `psi` and `w_rbf`) — that extra kept-and-shared axis is
  why the output is `boi` and not just `bo`.

So `edge[b,o,i] = \sum_k psi[b,i,k] * w_rbf[o,i,k]`: for each (trajectory `b`, input `i`)
it is a length-`k` dot product against every output `o`. The next sections build the real
`psi` and `w_rbf` and prove this.


## 2 · The math these letters encode (KAN edge functions)

Now that the letters are pinned down, here is *why* the contraction is over `k`. A KAN
replaces every scalar weight with a learned **univariate function on an edge**. For input
feature `i` feeding output node `o`, that edge function is a sum of `K` Gaussian bumps
(ChemKAN Eq. 11–12):

$$\phi_{o,i}(x_i)\;=\;\sum_{k=1}^{K} w^{\psi}_{o,i,k}\;\psi_k(x_i),
\qquad \psi_k(x_i)=\exp\!\Big(-\tfrac{(x_i-c_k)^2}{2h^2}\Big)$$

The sum $\sum_k$ **is** the contraction. Line the tensors up with the letters:

- `psi[b, i, k]`  = bump $k$ evaluated at input $i$ of trajectory $b$   (the $\psi_k(x_i)$)
- `w_rbf[o, i, k]` = weight of bump $k$ on the edge input $i\!\to\!$ output $o$   (the $w^{\psi}_{o,i,k}$)
- `edge[b, o, i]` = $\phi_{o,i}(x_i)$ = the edge value

So `"bik,oik->boi"` reads: **multiply, sum away the bump index `k`, keep `b,o,i`** — which
is exactly the equation above evaluated for every trajectory, output, and input at once.


In [47]:
import numpy as np
import torch

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

# --- load 3 real biodiesel trajectories at t=0 -> the KAN input x = [Y_1..Y_6, T] ---
data = np.load("../data/generated/biodiesel.npz", allow_pickle=True)
states = torch.as_tensor(data["train_states"], dtype=torch.float32)  # (cases, T, m)
T_const = torch.as_tensor(data["train_T"], dtype=torch.float32)      # (cases,)
species = [str(s) for s in data["species"]]

B = 2                                        # take the first 2 trajectories
Y0 = states[:B, 0, :]                        # (2, 6) species mass fractions at t=0
T0 = T_const[:B].unsqueeze(-1)               # (2, 1) each trajectory's temperature
x = torch.cat([Y0, T0], dim=-1)              # (2, 7)  the state u = [Y, T]

print("species:", species)
print("x = [Y, T] shape:", tuple(x.shape))
print(x)


species: ['TG', 'ROH', 'DG', 'MG', 'GL', 'RCO2R']
x = [Y, T] shape: (2, 7)
tensor([[  1.4554,   0.5425,   0.0000,   0.0000,   0.0000,   0.0000, 334.4306],
        [  0.9047,   0.6864,   0.0000,   0.0000,   0.0000,   0.0000, 329.4374]])


## 2 · Build `psi` and `w_rbf` exactly like `RBFEdgeFunctions`

Two things happen before the einsum:

1. **`tanh` the input** — the KAN's *internal* normalization (distinct from the
   dataset min-max) so every value lands on the fixed center grid $[-1,1]$. Raw
   temperature (~330 K) saturates here — that is the whole reason the dynamics layer
   min-max scales the state first — but for this notebook we look at the raw path so
   the saturation is visible.
2. **evaluate the Gaussian bumps** $\psi_k$ → this is the `psi` tensor.


In [50]:
x

tensor([[  1.4554,   0.5425,   0.0000,   0.0000,   0.0000,   0.0000, 334.4306],
        [  0.9047,   0.6864,   0.0000,   0.0000,   0.0000,   0.0000, 329.4374]])

In [48]:
IN, OUT_DIM, K = 7, 4, 3          # in_features (m+1), hidden_dim, num_basis  (biodiesel)

# fixed center grid on [-1, 1] and the bump width h (spacing between centers)
centers = torch.linspace(-1.0, 1.0, K)                 # (K,)
h = (1.0 - (-1.0)) / (K - 1)                            # scalar float
print("centers:", centers, " h =", h)

def gaussian(x, centers, h):
    r = x.unsqueeze(-1) - centers                      # (..., K) broadcast x vs centers
    return torch.exp(-r ** 2 / (2 * h ** 2))

xn = torch.tanh(x)                                     # (2, 7) internal normalization
psi = gaussian(xn, centers, h)                         # (2, 7, 3) = (B, in, K)
w_rbf = torch.randn(OUT_DIM, IN, K) * 0.1              # (4, 7, 3) = (out, in, K)

print("psi   (b,i,k):", tuple(psi.shape))
print("w_rbf (o,i,k):", tuple(w_rbf.shape))


centers: tensor([-1.,  0.,  1.])  h = 1.0
psi   (b,i,k): (2, 7, 3)
w_rbf (o,i,k): (4, 7, 3)


In [74]:
states.shape

torch.Size([20, 30, 6])

In [73]:
gaussian(states[1,...], centers, h)

tensor([[[0.1630, 0.6642, 0.9955],
         [0.2412, 0.7901, 0.9520],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065]],

        [[0.1684, 0.6744, 0.9937],
         [0.2485, 0.7997, 0.9466],
         [0.5967, 0.9999, 0.6163],
         [0.6060, 1.0000, 0.6070],
         [0.6065, 1.0000, 0.6065],
         [0.5957, 0.9998, 0.6173]],

        [[0.1736, 0.6841, 0.9918],
         [0.2563, 0.8095, 0.9406],
         [0.5883, 0.9995, 0.6248],
         [0.6047, 1.0000, 0.6083],
         [0.6065, 1.0000, 0.6066],
         [0.5845, 0.9993, 0.6286]],

        [[0.1787, 0.6933, 0.9897],
         [0.2645, 0.8195, 0.9342],
         [0.5809, 0.9991, 0.6321],
         [0.6027, 1.0000, 0.6103],
         [0.6064, 1.0000, 0.6067],
         [0.5729, 0.9985, 0.6401]],

        [[0.1836, 0.7020, 0.9875],
         [0.2729, 0.8294, 0.9274],
         [0.5746, 0.9986, 0.6384],
         [0.6003, 0.9999, 0.6128],
         [0.

In [115]:
# first trajectory
states[0, :].shape
# First training trajectory.
# 30 rows = 30 sampled time points over the 30-second trajectory.
# IMPORTANT: this does NOT necessarily mean one row per second.
# 6 columns = the 6 biodiesel species; temperature is stored separately.

torch.Size([30, 6])

In [116]:
# State of the 6 species at the initial recorded time t0
# for trajectory 0.
#
# If we append temperature, this gives the initial ChemKAN input u(t0).
#
# For this notebook example, we evaluate the architecture at this state.
# During actual Neural-ODE integration, however, the ODE solver can call
# ChemKAN many times at its own internal time points, not only at these
# 30 stored observation times.
states[0,0,:]

tensor([1.4554, 0.5425, 0.0000, 0.0000, 0.0000, 0.0000])

In [111]:
states[0,0,:].unsqueeze(-1), centers

(tensor([[1.4554],
         [0.5425],
         [0.0000],
         [0.0000],
         [0.0000],
         [0.0000]]),
 tensor([-1.,  0.,  1.]))

In [114]:
states[0,0,:].unsqueeze(-1) - centers

tensor([[ 2.4554,  1.4554,  0.4554],
        [ 1.5425,  0.5425, -0.4575],
        [ 1.0000,  0.0000, -1.0000],
        [ 1.0000,  0.0000, -1.0000],
        [ 1.0000,  0.0000, -1.0000],
        [ 1.0000,  0.0000, -1.0000]])

In [ ]:
# in gaussian function, states vectors each value converrted to a vector, so it became a column vector
states[0,0,:].unsqueeze(-1) - centers
# this difference calculated as for each value of the centers subtracted from each value of the column vector 
# and each row became 1x3 vector
# For each scalar x_i​, subtract each center c_k​ from that scalar.
# as in the notation \psi(||x-c_i||)

tensor([[ 2.4554,  1.4554,  0.4554],
        [ 1.5425,  0.5425, -0.4575],
        [ 1.0000,  0.0000, -1.0000],
        [ 1.0000,  0.0000, -1.0000],
        [ 1.0000,  0.0000, -1.0000],
        [ 1.0000,  0.0000, -1.0000]])

In [117]:
# we first normalize this trajectory values states[0,0,:] with tanh then we put this to gaussian function to calculate it
gaussian(torch.tanh(states[0,0,:]), centers, h)
# TG      [0.1655, 0.6689, 0.9947]
# ROH     [0.3272, 0.8848, 0.8802]
# DG      [0.6065, 1.0000, 0.6065]
# MG      [0.6065, 1.0000, 0.6065]
# GL      [0.6065, 1.0000, 0.6065]
# RCO2R   [0.6065, 1.0000, 0.6065]

tensor([[0.1655, 0.6689, 0.9947],
        [0.3272, 0.8848, 0.8802],
        [0.6065, 1.0000, 0.6065],
        [0.6065, 1.0000, 0.6065],
        [0.6065, 1.0000, 0.6065],
        [0.6065, 1.0000, 0.6065]])

In [ ]:
# these steps are done for all batches(trajectories) and we got psi

In [63]:
psi

tensor([[[0.1655, 0.6689, 0.9947],
         [0.3272, 0.8848, 0.8802],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065],
         [0.1353, 0.6065, 1.0000]],

        [[0.2284, 0.7725, 0.9612],
         [0.2800, 0.8374, 0.9215],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065],
         [0.6065, 1.0000, 0.6065],
         [0.1353, 0.6065, 1.0000]]])

In [64]:
# i guess since we have 3 bumps and we have 7 features and we have these bumps for each input feature so we have these for each batch separately

Notice `tanh(330 K) == 1.0` for the temperature column — all three trajectories
collapse to the same value there. That is the saturation the model's input-scaling
fix addresses; here it just makes the einsum inputs concrete.


In [49]:
print("raw T column of x :", x[:, -1])
print("after tanh (col -1):", xn[:, -1], " <- all 1.0, saturated")


raw T column of x : tensor([334.4306, 329.4374])
after tanh (col -1): tensor([1., 1.])  <- all 1.0, saturated


## 3 · The einsum — then re-derive it three ways

`edge[b,o,i] = sum_k psi[b,i,k] * w_rbf[o,i,k]`.


In [97]:
edge = torch.einsum("bik,oik->boi", psi, w_rbf)   # (2, 4, 7) = (B, out, in)
print("edge (b,o,i):", tuple(edge.shape))
print(edge[0])                                    # the 4x7 grid of edge values for traj 0


edge (b,o,i): (2, 4, 7)
tensor([[-0.1206,  0.1218, -0.2111, -0.0229,  0.1988, -0.2531,  0.1157],
        [ 0.1357, -0.0425,  0.2463, -0.0642, -0.0323,  0.0768,  0.0790],
        [ 0.0507, -0.1943,  0.1105, -0.0360, -0.0336,  0.4041,  0.0864],
        [ 0.0443,  0.2246,  0.1014,  0.0172, -0.1386, -0.0201,  0.2155]])


In [145]:
sum = torch.tensor(0, dtype=torch.float32)
for i in range(7):
    sum += psi[0,i,:] @ w_rbf[2,i,:]
    print(psi[0,i,:] @ w_rbf[2,i,:])
# this give us the \alpha = 3, \beta \in [1,7] 
print("sum:",sum)

tensor(0.0507)
tensor(-0.1943)
tensor(0.1105)
tensor(-0.0360)
tensor(-0.0336)
tensor(0.4041)
tensor(0.0864)
sum: tensor(0.3877)



### 3a · By hand: use the **same paper edge** $\phi_{0,3,2}$

The paper uses one-based mathematical indexing. PyTorch uses zero-based indexing.

Therefore the paper edge

$$
\boxed{\phi_{0,3,2}}
$$

(destination/hidden node $\alpha=3$, source node $\beta=2=\mathrm{ROH}$)

corresponds in PyTorch to

```python
o = 2   # paper alpha = 3
i = 1   # paper beta  = 2
```

For trajectory `b=0`, the edge value is

$$
E^{(0)}_{0,3,2}
=
\sum_{k=1}^{3}
\psi_{0,2,k}\,w^\psi_{0,3,2,k}.
$$

In zero-based code this is `edge[0, 2, 1]`.


In [140]:

b = 0

# Paper indexing: alpha=3, beta=2
# PyTorch indexing: o=2, i=1
o, i = 2, 1

manual_scalar = sum(psi[b, i, k] * w_rbf[o, i, k] for k in range(K))

print("Paper edge      : phi_{0,3,2}  = ROH -> hidden node 3")
print("PyTorch entry   : edge[0, 2, 1]")
print("einsum edge val :", edge[b, o, i].item())
print("manual edge val :", manual_scalar.item())
print("psi[b,i,:]      :", psi[b, i])
print("w_rbf[o,i,:]    :", w_rbf[o, i])

assert torch.allclose(edge[b, o, i], manual_scalar)


Paper edge      : phi_{0,3,2}  = ROH -> hidden node 3
PyTorch entry   : edge[0, 2, 1]
einsum edge val : -0.19430597126483917
manual edge val : -0.19430597126483917
psi[b,i,:]      : tensor([0.3272, 0.8848, 0.8802])
w_rbf[o,i,:]    : tensor([ 0.2302, -0.1469, -0.1587])


### 3a.1 · From that one edge to the complete hidden node 3

The scalar above is **only one of the seven incoming edges** to hidden node 3.

For the same trajectory,

$$
x_{1,3}
=
\sum_{\beta=1}^{7}
\phi_{0,3,\beta}(x_{0,\beta}).
$$

In the evaluated edge tensor, paper row $\alpha=3$ is PyTorch row `o=2`:

```python
edge[b, 2, :]
```

That row contains seven numbers:

$$
[
\phi_{0,3,1}(x_{0,1}),\ldots,\phi_{0,3,7}(x_{0,7})
].
$$

Summing the row gives the value of hidden node 3.


In [146]:
# The 7 evaluated incoming edges for paper hidden node alpha=3
incoming_to_hidden3 = edge[b, 2, :]

# AddKAN row reduction: sum over source-node index beta
hidden3_manual = incoming_to_hidden3.sum()

# All four hidden nodes at once
hidden = edge.sum(dim=-1)

print("7 incoming edge values to hidden node 3:")
print(incoming_to_hidden3)
print("\nhidden node 3 from row sum :", hidden3_manual.item())
print("hidden[0, 2]                :", hidden[b, 2].item())

assert torch.allclose(hidden3_manual, hidden[b, 2])


7 incoming edge values to hidden node 3:
tensor([ 0.0507, -0.1943,  0.1105, -0.0360, -0.0336,  0.4041,  0.0864])

hidden node 3 from row sum : 0.3877328038215637
hidden[0, 2]                : 0.3877328038215637


### 3b · Full triple loop (the einsum spelled out)


In [147]:
edge_loop = torch.zeros(B, OUT_DIM, IN)
for bb in range(B):
    for oo in range(OUT_DIM):
        for ii in range(IN):
            edge_loop[bb, oo, ii] = (psi[bb, ii, :] * w_rbf[oo, ii, :]).sum()  # sum over k

print("max |einsum - loop| =", (edge - edge_loop).abs().max().item())
assert torch.allclose(edge, edge_loop, atol=1e-6)


max |einsum - loop| = 1.4901161193847656e-08


### 3c · Broadcasting and matmul (vectorized, no einsum)

The einsum is *not* magic — it is a broadcasted multiply and a sum over `k`. Align the
shapes by inserting the missing axes:

- `psi`   is `(b, i, k)` → view as `(b, 1, i, k)` (broadcast over `o`)
- `w_rbf` is `(o, i, k)` → view as `(1, o, i, k)` (broadcast over `b`)


In [148]:
edge_bcast = (psi[:, None, :, :] * w_rbf[None, :, :, :]).sum(dim=-1)   # sum over k
assert torch.allclose(edge, edge_bcast, atol=1e-6)

# Equivalently: for each input i it is a matmul  psi[:, i, :] @ w_rbf[:, i, :].T
edge_mm = torch.stack([psi[:, i, :] @ w_rbf[:, i, :].T for i in range(IN)], dim=-1)
assert torch.allclose(edge, edge_mm, atol=1e-6)
print("einsum == broadcast == matmul  ✓  (three independent derivations agree)")


einsum == broadcast == matmul  ✓  (three independent derivations agree)


> **Reading `"bik,oik->boi"` in one breath:** the letter missing from the output
> (`k`) is *summed over*; every letter kept on the right is a free axis of the result.
> A letter shared by both inputs but kept in the output (`i`) is a *batched* axis — it
> lines up element-for-element and is **not** contracted. That single distinction
> (shared-and-dropped `k` vs shared-and-kept `i`) is the whole trick.


## 4 · What the layers do with `edge`

`RBFEdgeFunctions` returns `edge` of shape `(B, out, in)`. The layers differ only in
how they collapse the **input axis** `i`:


In [149]:
# AddKANLayer:  y_o = sum_i edge[b, o, i]      (ChemKAN Eq. 7)
add_out = edge.sum(dim=-1)                        # (B, out)
print("AddKAN out (B,out):", tuple(add_out.shape))

# LeanKANLayer with n_mu = 2: multiply first 2 inputs, add the rest (LeanKAN Eq. 8-10)
n_mu = 2
mult = edge[..., :n_mu].prod(dim=-1)              # product over first n_mu inputs
addv = edge[..., n_mu:].sum(dim=-1)              # sum over the remaining inputs
lean_out = mult + addv
print("LeanKAN out (B,out):", tuple(lean_out.shape))
print("\nrow 0  add :", add_out[0])
print("row 0 lean :", lean_out[0])


AddKAN out (B,out): (2, 4)
LeanKAN out (B,out): (2, 4)

row 0  add : tensor([-0.1716,  0.3988,  0.3877,  0.4444])
row 0 lean : tensor([-0.1874,  0.2998,  0.5215,  0.1854])


And the optional Swish **base path** (ChemKAN Eq. 11, `use_base_act=True`) is a
*second, smaller* einsum with no `k` axis — one scalar weight per edge instead of `K`:

```python
edge = edge + torch.einsum("bi,oi->boi", base(x), w_base)   # "bi,oi->boi"
```

Same reading: nothing is contracted (there is no repeated-and-dropped letter), so it is
a pure outer-product-style broadcast placing `base(x)[b,i] * w_base[o,i]` at `[b,o,i]`.
Our main reproduction keeps this **off** to match the paper's 156/344 parameter counts.


## Takeaways

- The **RBF edge-evaluation step** is one contraction: `"bik,oik->boi"` =
  $\phi_{o,i}(x_i)=\sum_k w_{o,i,k}\psi_k(x_i)$.
- **`k`** (basis) is summed away; **`b`** (batch) and **`o`** (output) are free; **`i`**
  (input) is a *batched, kept* axis — the source of the `boi` layout.
- `AddKAN` = sum edges over `i`; `LeanKAN` = product of the first `n_mu` + sum of the rest.
- `tanh` saturates on raw Kelvin (all three trajectories collapsed to 1.0), which is why
  the dynamics layer min-max scales the physical state *before* the KAN.

Next: **`03_chemkan_from_scratch.ipynb`** builds these layers into the full ChemKAN
neural-ODE and trains it on biodiesel.
